# POSE — `infer.ipynb` · Bild rein → 3D-Viewer raus

Dieses Notebook von **oben nach unten** durchlaufen, um aus einem 2D-Bild eine
schema-valide `pose_result.json` zu erzeugen und sie im **localhost-3D-Viewer**
anzusehen. Es importiert die getestete Pipeline aus `e2e_infer.py` +
`bop_adapter.py` — **keine Code-Duplikate**.

## Pipeline (ADR-018, Viktor §3/§4)

```
Bild (input/) ─Detektor(YOLOv8-OBB)─▶ OBB→AABB-Crops
              ─GDRNPP(RGB-only)─▶ (R_m2c, t_m2c) [BOP, mm, Kamera-Frame]
              ─bop_adapter §3─▶ Welt-Pose (R_world, t_world, face, upright)
              ─▶ temp/pose_result.json  (Contract: pose_result.schema.json)
frontend/     pose_result.json + cell.glb ─▶ Three.js-Viewer (echtes CAD @ 6D-Pose)
```

## Läuft JETZT schon — MOCK-Pose-Backend

Solange Kais GDRNPP-Checkpoint noch trainiert, läuft Stufe 3 im **MOCK-Modus**:
`e2e_infer.call_gdrnpp` liefert pro Detektion deterministische, plausible Posen,
sodass die **ganze Kette inkl. Viewer JETZT** grün durchläuft. Ist ein echtes
`.pth` da, setzt du unten `GDRNPP_CHECKPOINT` und die Kette schaltet automatisch
auf den echten Call (Einhänge-Punkt `e2e_infer._gdrnpp_real`).

> RGB-only **hart** (ADR-018). Konvention: Z-up Welt, `world = R @ body`,
> Ursprung = Tisch-Nullpunkt, Einheit Meter.

**Abhängigkeiten (lokal):** `numpy` + `Pillow` (Pflicht). `jsonschema` ist
optional (Bonus-Schema-Gate; ohne fällt `e2e_infer` auf das stdlib-Gate).
`ultralytics`/`torch` nur für den echten Detektor-Checkpoint (sonst Fallback).

## 1 · Eingabebild laden (aus `project/input/`)

Lege dein Szenenbild nach `project/input/` (z.B. `scene_0000.png`). Liegt eine
`bbox_2d_*.json` (SDG-Annotator-Format) oder `scene_camera.json` daneben,
werden sie automatisch genutzt. Ist `input/` leer, erzeugt die nächste Zelle
ein synthetisches Demo-Bild, damit das Notebook **ohne manuelles Setup** läuft.

In [ ]:
import pathlib, sys, json
import numpy as np
from PIL import Image, ImageDraw, ImageFont

# Projekt-Root finden (CWD-unabhängig) + Pipeline importieren (kein Copy).
PROJECT = pathlib.Path.cwd()
if not (PROJECT / 'e2e_infer.py').exists():
    PROJECT = next((p for p in [PROJECT/'project', *PROJECT.parents]
                    if (p/'e2e_infer.py').exists()), PROJECT)
sys.path.insert(0, str(PROJECT))
import e2e_infer as E       # getestete Pipeline (Detektor->GDRNPP->Adapter->Contract)
import bop_adapter as BOP   # BOP->pose_result-Adapter (Viktor §3)

INPUT_DIR = PROJECT / 'input'
TEMP_DIR  = PROJECT / 'temp'; TEMP_DIR.mkdir(exist_ok=True)
print('PROJECT:', PROJECT)
print('obj_id-Map:', BOP.OBJ_ID_TO_PART)

In [ ]:
def find_input_image():
    """Erstes Bild in input/ — sonst ein synthetisches Demo-Bild erzeugen."""
    imgs = sorted([p for p in INPUT_DIR.glob('*')
                   if p.suffix.lower() in ('.png', '.jpg', '.jpeg')])
    if imgs:
        return imgs[0], False
    # Synthetisches Demo: grauer Hintergrund + 3 farbige Teil-Blobs + bbox-JSON.
    W, H = 1280, 720
    rng = np.random.default_rng(7)
    arr = rng.integers(45, 80, size=(H, W, 3)).astype(np.uint8)
    demo = Image.fromarray(arr); d = ImageDraw.Draw(demo)
    blobs = [((300, 200, 360, 480), (210, 90, 90), 'anker_kurz', 0),
             ((700, 150, 760, 470), (90, 160, 210), 'anker_lang', 1),
             ((540, 320, 640, 420), (120, 200, 120), 'zahnrad',    5)]
    rows = []
    for (x0, y0, x1, y1), col, _, cls in blobs:
        d.ellipse([x0, y0, x1, y1], fill=col)
        rows.append([cls, x0, y0, x1, y1, 0.0])
    img_path = INPUT_DIR / 'scene_demo.png'; demo.save(img_path)
    bbox_doc = {'data': rows, 'info': {'idToLabels': {
        '0': {'class': 'anker_kurz'}, '1': {'class': 'anker_lang'},
        '5': {'class': 'zahnrad'}}}}
    json.dump(bbox_doc, open(INPUT_DIR / 'bbox_2d_demo.json', 'w'))
    return img_path, True

IMAGE, is_demo = find_input_image()
print(('DEMO-Bild erzeugt: ' if is_demo else 'Eingabebild: ') + str(IMAGE))
Image.open(IMAGE).convert('RGB')

## 2 · Detektor → OBB→AABB

`e2e_infer.detections_for` liefert Detektionen in dieser Priorität:
1. echter **YOLOv8-OBB-Detektor** (`models/detector.pt`, OBB→AABB intern),
2. **SDG-Annotator-Boxen** (`bbox_2d_*.json` neben dem Bild),
3. **Dummy** (eine Ganzbild-Box) — die Kette bricht nie hart ab.

So läuft die Stufe sofort, auch ohne trainierten Detektor-Checkpoint.

In [ ]:
rgb, dets = E.detections_for(IMAGE)
print(f'{len(dets)} Detektion(en):')
for d in dets:
    print(f"  #{d['instance_id']:>2} {d['part']:<22} bbox={d['bbox_2d']}")

In [ ]:
# Detektionen aufs Bild zeichnen (PIL — immer verfügbar).
def draw_detections(rgb, dets):
    im = Image.fromarray(rgb).convert('RGB'); dr = ImageDraw.Draw(im)
    for d in dets:
        x0, y0, x1, y1 = d['bbox_2d']
        dr.rectangle([x0, y0, x1, y1], outline=(255, 80, 80), width=3)
        dr.text((x0 + 3, max(0, y0 - 14)), f"{d['instance_id']}:{d['part']}",
                fill=(255, 220, 0))
    return im

det_vis = draw_detections(rgb, dets)
det_vis.save(TEMP_DIR / 'detections.png')
print('Overlay ->', TEMP_DIR / 'detections.png')
det_vis

## 3 · GDRNPP-Inferenz (Checkpoint-Variable; MOCK bis Training fertig)

Setze `GDRNPP_CHECKPOINT` auf den trainierten `.pth`-Pfad, sobald
`setup.ipynb` Stufe 6 (`train_chain.sh`) `TRAIN_CHAIN_DONE` meldet. Fehlt der
Checkpoint, läuft alles **automatisch im MOCK-Modus** — klar markiert.

`e2e_infer.GdrnppConfig` entscheidet selbst: existiert kein Checkpoint →
`mock=True`. `e2e_infer.estimate_poses` führt pro Detektion
`call_gdrnpp` → `bop_adapter.detection_to_result` aus (DER eine Adapter,
beide Gleise).

In [ ]:
# >>> Nach dem Training hier den echten Checkpoint-Pfad eintragen. <<<
GDRNPP_CHECKPOINT = None    # z.B. '/mnt/data/bop/repos/gdrnpp/output/.../anker_kurz/model_final.pth'

cfg = E.GdrnppConfig(checkpoint=GDRNPP_CHECKPOINT)
MODE = 'MOCK (plausible Posen — Viewer-tauglich)' if cfg.mock else 'GDRNPP (echter Checkpoint)'
print('Pose-Backend:', MODE)

# Kamera (BOP §1.3): scene_camera.json neben dem Bild oder Default-Top-Down.
H, W = rgb.shape[:2]
K, R_w2c, t_w2c = E.load_scene_camera(IMAGE, W, H)
E._SCENE_CAM_STATE.update({'K': K, 'R_w2c': R_w2c, 't_w2c': t_w2c})

aligned = E.estimate_poses(rgb, dets, cfg=cfg)
print(f'{len(aligned)} Pose(n) geschätzt (Backend={"MOCK" if cfg.mock else "GDRNPP"}).')

## 4 · `bop_adapter` → `pose_result.json` (schema-validiert)

`e2e_infer.build_pose_result` baut das Contract-Dokument; `e2e_infer.run`
kapselt die ganze Kette **und** das Schema-Gate (stdlib immer +
`jsonschema` wenn installiert). Es schreibt nur, wenn das Ergebnis
schema-valide ist — sonst `ValueError`.

In [ ]:
OUT = TEMP_DIR / 'pose_result.json'
doc = E.run(str(IMAGE), str(OUT), cfg=cfg)   # Detektor->GDRNPP->Adapter->Contract+Gate
print('\npose_result geschrieben + schema-valide ->', OUT)
print('Schema-Gate:', 'stdlib + jsonschema' if E.jsonschema_available() else 'stdlib (jsonschema optional)')

In [ ]:
# pose_result-Tabelle (eine Zeile pro Teil).
def show_pose_table(doc):
    print(f"{'#':>2}  {'part':<22} {'face':<10} {'conf':>5}  {'t_world (m)':<24} upright")
    print('-' * 78)
    for r in doc['results']:
        t = '[' + ', '.join(f'{v:+.3f}' for v in r['t_world']) + ']'
        print(f"{r['instance_id']:>2}  {r['part']:<22} {r['face']:<10} "
              f"{r['confidence']:>5.2f}  {t:<24} {r['upright']}")

show_pose_table(doc)
print('\nmeta:', json.dumps(doc['meta'], indent=2))

## 5 · (Optional) BOP-Eval gegen synth-Holdout

Sobald GDRNPP echte Posen liefert, scort `box_src/eval_bop.py` sie
**symmetrie-bewusst** mit den offiziellen `bop_toolkit`-Metriken (AR =
MSSD/MSPD, ADD/ADI). Headline ist `AR = mean(AR_MSSD, AR_MSPD)` (beide exakt
+ symmetrie-aware; VSD ist auf diesem Datensatz nur ein relatives Signal —
siehe `box_src/EVAL_BOP.md`).

Eval läuft **auf der Box** (braucht `bop_toolkit_lib`). Vom Laptop via Wrapper:

```bash
# Selbst-Test (kein Checkpoint nötig — beweist die Metrik-Mechanik):
box_src/eval_bop.sh --self-test

# Echte Predictions scoren (BOP-results-CSV auf der Box):
box_src/eval_bop.sh --preds /mnt/data/bop/results/gdrnpp/preds.csv
```

Der Wrapper holt `report.json`/`report.txt` nach `./results/eval/`. Dieses
Inferenz-Notebook erzeugt **eine** Szene für den Viewer; die Quantifizierung
über den ganzen `val`-Split macht der Eval-Harness.

## 6 · 3D-Viewer starten (ein Klick)

Die letzte Zelle startet einen localhost-Server ab `project/` und gibt den
Viewer-Link aus. Der Three.js-Viewer (`frontend/`) lädt `temp/pose_result.json`
und rendert die **echten CAD-Meshes** an ihrer 6D-Pose in der echten Anlage
(`frontend/assets/cell.glb`: Tisch + Roboterarm + Teile).

In [ ]:
from IPython.display import IFrame, display, HTML
import threading, functools, http.server, socketserver

PORT = 8000
rel  = OUT.resolve().relative_to(PROJECT)               # temp/pose_result.json
VIEWER_URL = f'http://127.0.0.1:{PORT}/frontend/?file=../{rel.as_posix()}'

def start_server(port=PORT, directory=str(PROJECT)):
    """Hintergrund-HTTP-Server ab project/. Idempotent (Port belegt -> reuse)."""
    handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=directory)
    socketserver.TCPServer.allow_reuse_address = True
    try:
        httpd = socketserver.TCPServer(('127.0.0.1', port), handler)
    except OSError:
        print(f'Port {port} bereits belegt — vermutlich läuft der Server schon.')
        return None
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
    print(f'Server läuft ab {directory} auf Port {port}.')
    return httpd

_httpd = start_server()
print('\nViewer-URL:', VIEWER_URL)
display(HTML(f'<a href="{VIEWER_URL}" target="_blank" style="font-size:16px">▶ 3D-Viewer in neuem Tab öffnen</a>'))

In [ ]:
# Viewer direkt im Notebook einbetten (falls die IFrame-Sandbox es zulässt;
# sonst den Link oben in neuem Tab öffnen).
IFrame(VIEWER_URL, width='100%', height=620)

### Alternativer Start (ohne Notebook)

Ein einziger Befehl vom Repo-Root — erzeugt `pose_result` **und** öffnet den Viewer:

```bash
python project/e2e_infer.py --image project/input/scene_demo.png --serve
```

Mit echtem Checkpoint:

```bash
python project/e2e_infer.py --image project/input/<bild>.png \
    --checkpoint /pfad/zu/gdrnpp_model.pth --serve
```